### SOAL 01 - Konsep STKI & Perkembangan (15%, Sub-CPMK10.1.1)

#### 1. Definisi STKI, beda dengan database retrieval, peran index & ranking.

   **Jawab:**

   - STKI / *Information Retrieval* merupakan sistem yang berfungsi untuk mengekstrak informasi dari suatu kumpulan data tidak terstruktur.
   - Perbedaan utama antara STKI dengan database retrieval ada 2:

      a. Data yang digunakan di STKI adalah data tidak terstruktur, sedangkan database retrieval pasti menggunakan data terstruktur (karena skemanya jelas).

      b. Dalam STKI, yang diambil adalah informasi (makna yang terkandung dalam suatu data), sedangkan database retrieval mengambil raw data.
   
   - **Index** berperan dalam mempercepat pemrosesan pengambilan informasi.
   
   - **Ranking** berperan dalam mengukur relevansi antara apa yang ingin didapat dengan apa yang ditemukan dalam kumpulan data.

#### 2. Garis besar arsitektur search engine klasik

![Search Engine](images/search-engine.png)

   **Jawab:**
   - Secara terus-menerus, web crawler dan indexing module terus bekerja dalam pembuatan index dalam server search engine.
   - Ketika user melakukan sebuah pencarian (mungkin lewat browser), query tersebut akan dikirim ke search engine dan akan diolah terlebih dahulu (preprocessing query)
   - Search engine akan melakukan perhitungan untuk mendapatkan hasil yang relevan berdasarkan parsed query dan index saat itu.
   - Setelah itu hasil akan diranking dan akhirnya dikirimkan kepada user.

#### 3. Sketsa arsitektur: Retrieval klasik (Boolean)

![Boolean IR](images/boolean-ir.png)

**Jawab:**

- Secara garis besar, arsitektur tersebut mirip dengan arsitektur search engine karena search engine juga merupakan retrieval system
- Yang jadi pembeda dengan Boolean IR ini adalah bahwa tidak adanya sistem perankingan dokumen. Hal ini karena karakteristik algoritma yang dipakai yang bersifat relevan atau tidak (tidak ada tengah-tengah)

#### 4. Peta materi ke RPS

![Peta RPS](images/peta-rps.png)

**Jawab:**

- Mapping sebenarnya sudah ada di situ, namun di sini saya akan mencoba menjelaskannya lebih detail.
- Soal 02 mengenai preprocessing. Intinya adalah pemrosesan dokumen dari raw data menjadi kumpulan token yang bersih (lowercase, bebas stopwords, case-folding, dan stemming)
- Soal 03 mengenai implementasi algoritma Boolean untuk IR.
- Soal 04 mengenai implementasi algoritma VSM untuk IR.
- Soal 05 mengenai search engine.
- Saya pribadi kurang setuju denga tabel di atas bahwa soal 5 melingkupi materi 5,6,7. Hal ini karena di soal tidak ada instruksi bahwa perlu menggunakan Word Embedding. Materi 5 dan 6 dimana keduanya mengenai Word Embedding tidak diuji di soal.   

# ===== Practical area below =====

In [1]:
import importlib
import glob
import os
import sys
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.path.abspath("../src"))
import preprocess
import boolean_ir
import vsm_ir
import eval

importlib.reload(preprocess)
importlib.reload(boolean_ir)
importlib.reload(vsm_ir)
importlib.reload(eval)

<module 'eval' from 'd:\\UrusanKuliah\\Perkuliahan\\Semester_7\\STKI\\stki-uts-A11202214623-StevenAdiSuryanto\\src\\eval.py'>

### SOAL 02 - Document Preprocessing (20%, Sub-CPMK10.1.2)

#### 1. Load + Preprocessing data

In [2]:
cleaned_docs = {}
raw_docs = {}
file_folder = "../data/*.txt"
for file in glob.glob(file_folder):
    filename = os.path.basename(file)
    file = open(file, "r", encoding="utf-8")
    text = file.read()

    raw_docs[filename] = text

    cleaned_text = preprocess.clean(text)
    tokens = preprocess.tokenize(cleaned_text)
    tokens_no_stopwords = preprocess.remove_stopwords(tokens)
    tokens_stemmed = preprocess.stems(tokens_no_stopwords)
    cleaned_docs[filename] = " ".join(tokens_stemmed)

cleaned_docs

{'cendrawasih.txt': 'cendrawasih ekor burung hutan hidup hutan papua',
 'gajah_asia.txt': 'gajah asia mamalia darat belalai gading hidup hutan tropis sabana asia selatan tenggara gajah sosial hidup kelompok pimpin betina tua milik ingat tajam nali manusia main peran jaga imbang ekosistem',
 'harimau.txt': 'harimau kucing mangsa hidup hutan asia bulu oranye gar hitam tubuh kuat harimau soliter aktif buru malam renang mangsa utama liput rusa babi hutan kerbau muda populasi turun buru liar hilang habitat',
 'katak.txt': 'katak pohon hijau milik kulit hijau lembap tubuh licin hidup pohon daerah lembap hutan hujan kebun katak aktif malam mangsa serangga lalat jangkrik katak pohon kenal tahan kering mudah pelihara tangkar',
 'komodo.txt': 'komodo kadal besar dunia endemik indonesia pulau komodo panjang capai meter berat kilogram komodo predator puncak makan rusa babi bangkai air liur kandung bakteri racun mati mangsa',
 'kuda_laut.txt': 'katak pohon hijau milik kulit hijau lembap tubuh licin

In [3]:
pd.set_option("display.max_colwidth", None)

raw_sample = list(raw_docs.items())[1:3]
cleaned_sample = list(cleaned_docs.items())[1:3]

two_cleaned_docs_df = pd.DataFrame(
    cleaned_sample, columns=["filename", "after"]
).set_index("filename")

two_cleaned_docs_df.insert(0, "before", [doc[1] for doc in raw_sample])
two_cleaned_docs_df

,before,after
filename,,
gajah_asia.txt,"Gajah Asia adalah mamalia darat besar dengan belalai panjang dan gading kecil. Mereka hidup di hutan tropis dan sabana Asia Selatan serta Tenggara. Gajah sangat sosial, hidup berkelompok dipimpin betina tertua. Mereka memiliki ingatan tajam, dapat mengenali manusia, dan memainkan peran penting menjaga keseimbangan ekosistem.",gajah asia mamalia darat belalai gading hidup hutan tropis sabana asia selatan tenggara gajah sosial hidup kelompok pimpin betina tua milik ingat tajam nali manusia main peran jaga imbang ekosistem
harimau.txt,"Harimau adalah kucing besar pemangsa yang hidup di hutan Asia. Bulunya oranye bergaris hitam dan tubuhnya sangat kuat. Harimau soliter, aktif berburu pada malam hari, dan dapat berenang dengan baik. Mangsa utamanya meliputi rusa, babi hutan, dan kerbau muda. Populasinya menurun karena perburuan liar dan kehilangan habitat.",harimau kucing mangsa hidup hutan asia bulu oranye gar hitam tubuh kuat harimau soliter aktif buru malam renang mangsa utama liput rusa babi hutan kerbau muda populasi turun buru liar hilang habitat


#### 2. Write hasil preprocessing ke /data/processed

In [4]:
for filename in cleaned_docs:
    with open("../data/processed/" + filename, "w") as f:
        f.write(cleaned_docs[filename])

### SOAL 03 - Boolean Retrieval Model (20%, Sub-CPMK10.1.3)

#### 1. Buat vocabullary

In [5]:
all_words = [word for doc in cleaned_docs.values() for word in preprocess.tokenize(doc)]

vocabullary = list(set(all_words))
vocabullary

['cokelat',
 'paus',
 'tahan',
 'jangkrik',
 'populasi',
 'tangkar',
 'racun',
 'biru',
 'hilang',
 'ratus',
 'kering',
 'mamalia',
 'bakteri',
 'kalimantan',
 'kerbau',
 'dagang',
 'tenggara',
 'oranye',
 'muda',
 'pulau',
 'air',
 'kucing',
 'predator',
 'deforestasi',
 'dengar',
 'jaga',
 'kadal',
 'punah',
 'status',
 'mati',
 'harimau',
 'sendiri',
 'komunikasi',
 'daerah',
 'tubuh',
 'orangutan',
 'kulit',
 'buru',
 'hitam',
 'sumatra',
 'main',
 'kera',
 'asia',
 'rusa',
 'nali',
 'imbang',
 'ton',
 'sabana',
 'kandung',
 'liur',
 'makan',
 'satwa',
 'habitat',
 'ekosistem',
 'kril',
 'gading',
 'berat',
 'mangsa',
 'babi',
 'pimpin',
 'turun',
 'gajah',
 'katak',
 'kenal',
 'cerdas',
 'lalat',
 'pohon',
 'alat',
 'puncak',
 'ajar',
 'suara',
 'kelompok',
 'hujan',
 'besar',
 'belalai',
 'aktif',
 'hijau',
 'ranting',
 'indonesia',
 'komodo',
 'bumi',
 'peran',
 'endemik',
 'papua',
 'laut',
 'panjang',
 'burung',
 'sosial',
 'rendah',
 'utama',
 'kilometer',
 'betina',
 'renang

#### 2a. Buat incidence matrix

In [6]:
incidence_matrix = boolean_ir.build_incidence_matrix(cleaned_docs)
print(incidence_matrix.shape)
incidence_matrix.toarray()

(129, 8)


array([[0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 1, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0]])

#### 2b. Buat inverted index

In [7]:
inverted_index = boolean_ir.build_inverted_index(cleaned_docs)
inverted_index

{'cendrawasih': [(0, 1, [0])],
 'ekor': [(0, 1, [1])],
 'burung': [(0, 1, [2])],
 'hutan': [(0, 2, [3, 5]),
  (1, 1, [7]),
  (2, 2, [4, 23]),
  (3, 1, [13]),
  (5, 1, [13]),
  (6, 1, [6])],
 'hidup': [(0, 1, [4]),
  (1, 2, [6, 15]),
  (2, 1, [3]),
  (3, 1, [9]),
  (5, 1, [9]),
  (6, 2, [5, 12]),
  (7, 1, [9])],
 'papua': [(0, 1, [6])],
 'gajah': [(1, 2, [0, 13])],
 'asia': [(1, 2, [1, 10]), (2, 1, [5])],
 'mamalia': [(1, 1, [2])],
 'darat': [(1, 1, [3])],
 'belalai': [(1, 1, [4])],
 'gading': [(1, 1, [5])],
 'tropis': [(1, 1, [8])],
 'sabana': [(1, 1, [9])],
 'selatan': [(1, 1, [11])],
 'tenggara': [(1, 1, [12])],
 'sosial': [(1, 1, [14])],
 'kelompok': [(1, 1, [16])],
 'pimpin': [(1, 1, [17])],
 'betina': [(1, 1, [18])],
 'tua': [(1, 1, [19])],
 'milik': [(1, 1, [20]), (3, 1, [3]), (5, 1, [3]), (6, 1, [17])],
 'ingat': [(1, 1, [21])],
 'tajam': [(1, 1, [22])],
 'nali': [(1, 1, [23])],
 'manusia': [(1, 1, [24])],
 'main': [(1, 1, [25])],
 'peran': [(1, 1, [26])],
 'jaga': [(1, 1, [27])

#### 3. Buat Boolean query parser

In [8]:
all_files = list(cleaned_docs.keys())
parser = boolean_ir.BooleanQueryParser(incidence_matrix, vocabullary, all_files)

print(parser.evaluate("NOT serangga"))

['cendrawasih.txt', 'gajah_asia.txt', 'harimau.txt', 'komodo.txt', 'orangutan.txt', 'paus_biru.txt']


#### 4. Evaluasi Boolean parser

In [9]:
queries = ["cendrawasih OR burung", "mangsa AND kerbau", "NOT serangga"]

gold_set = [
    ["cendrawasih.txt"],
    ["harimau.txt"],
    [
        "cendrawasih.txt",
        "gajah_asia.txt",
        "harimau.txt",
        "komodo.txt",
        "orangutan.txt",
        "paus_biru.txt",
    ],
]

retrieval_result = [parser.evaluate(query) for query in queries]


def precision_at_k(retrieved, relevant):
    retrieved_set = set(retrieved)
    relevant_set = set(relevant)
    if not retrieved_set:
        return 0.0
    return len(retrieved_set & relevant_set) / len(retrieved_set)


for i, (retrieved, relevant) in enumerate(zip(retrieval_result, gold_set), 1):
    p = eval.precision_at_k(retrieved, relevant)
    print(f"Query {i}:")
    print(f"  Precision: {p:.2f}")


Query 1:
  Precision: 1.00
Query 2:
  Precision: 1.00
Query 3:
  Precision: 1.00


### SOAL 04 - Vector Space Model & Ranking (20%, Sub-CPMK10.1.3)

#### 1. TF-IDF dari dokumen

In [10]:
vectorizer = TfidfVectorizer(ngram_range=(1, 1), smooth_idf=False)
tfidf_matrix = vectorizer.fit_transform(list(cleaned_docs.values()))
tfidf_matrix.shape

(8, 129)

#### 2. TF-IDF dari query

In [11]:
query = ["hutan penuh mangsa"]

print("Representasi matriks TF-IDF dari query:")
tf_idf_query = vectorizer.transform(query)
tf_idf_query[0].toarray()

Representasi matriks TF-IDF dari query:


array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.60534851, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

#### 3. Cosine similarity setiap dokumen terhadap query

In [12]:
doc_sim = cosine_similarity(tfidf_matrix, tf_idf_query)
doc_sim_df = pd.DataFrame(doc_sim)
doc_sim_df

,0
0,0.230239
1,0.046667
2,0.257040
3,0.139570
4,0.081119
5,0.139570
6,0.052972
7,0.000000


In [13]:
doc_sim.flatten()

array([0.23023857, 0.04666731, 0.25703953, 0.13956952, 0.08111879,
       0.13956952, 0.0529716 , 0.        ])

Ambil top-3 (query itu sendiri tidak termasuk)

In [20]:
top_3_idxs = np.argsort(-doc_sim.flatten())[:3]

top_3_docs = [all_files[i] for i in top_3_idxs]

rows = []
for i in top_3_idxs:
    doc_id = all_files[i]
    score = float(doc_sim.flatten()[i])
    snippet = cleaned_docs[doc_id][:120]

    rows.append({"doc_id": doc_id, "cosine_score": score, "snippet": snippet})

print("Query: " + query[0])
df = pd.DataFrame(rows)
df


Query: hutan penuh mangsa


,doc_id,cosine_score,snippet
0,harimau.txt,0.257040,harimau kucing mangsa hidup hutan asia bulu oranye gar hitam tubuh kuat harimau soliter aktif buru malam renang mangsa u
1,cendrawasih.txt,0.230239,cendrawasih ekor burung hutan hidup hutan papua
2,katak.txt,0.139570,katak pohon hijau milik kulit hijau lembap tubuh licin hidup pohon daerah lembap hutan hujan kebun katak aktif malam man


#### 4. Evaluasi VSM query engine dengan Precision@k

In [15]:
queries = ["hutan penuh mangsa"]
gold_set = [["harimau.txt", "cendrawasih.txt", "kuda_laut.txt"]]

vsm_parser = vsm_ir.VSMQueryParser(
    tfidf_matrix, vectorizer, list(cleaned_docs.keys()), k=3
)
retrieval_result = [vsm_parser.evaluate(query) for query in queries]


for i, (retrieved, relevant) in enumerate(zip(retrieval_result, gold_set), 1):
    retrieved = [rec[0] for rec in retrieved]
    p = eval.precision_at_k(retrieved, relevant)
    print(retrieved)
    print(f"Query {i}:")
    print(f"  Precision: {p:.2f}")


['harimau.txt', 'cendrawasih.txt', 'kuda_laut.txt']
Query 1:
  Precision: 1.00


### SOAL 05 - Term Weighting, Search Engine, dan Evaluasi (25%, Sub-CPMK10.1.4)

#### 1. Perbandingan skema term weighting (TF-IDF vs TF-IDF sublinear)

In [16]:
sublinear_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1), smooth_idf=False, sublinear_tf=True
)
sublinear_vectorizer.fit_transform(list(cleaned_docs.values()))

<8x129 sparse matrix of type '<class 'numpy.float64'>'
	with 184 stored elements in Compressed Sparse Row format>

#### 2.  Search Engine Orchestrator

In [ ]:
!python ../src/search_engine.py --model vsm --k 3 --query "hutan"


=== VSM SEARCH RESULTS ===
- cendrawasih.txt  (score=0.3803)
- harimau.txt  (score=0.1556)
- orangutan.txt  (score=0.0875)



In [18]:
!python ../src/search_engine.py --model boolean --query "hutan"


=== BOOLEAN SEARCH RESULTS ===
- cendrawasih.txt
- gajah_asia.txt
- harimau.txt
- katak.txt
- kuda_laut.txt
- orangutan.txt



#### 3. Main interface

In [19]:
# Jika tidak bisa dijalankan, jalankan script chat.py tersebut di CLI
!python ../app/chat.py 

^C
